# Pickle and Joblib

---

In this notebook, we will learn how to save (serialize) and load (deserialize) trained machine learning models using Python's two primary tools: **pickle** and **joblib**.

We will cover:
- Saving and loading a model with `pickle`
- Saving and loading a model with `joblib` (and why it's preferred for ML)
- Comparing file sizes with and without compression
- The critical practice of persisting **entire Pipelines**, not just bare models
- Saving model metadata alongside the artifact
- Verifying that a loaded model produces identical predictions

---

## 1. Setup
Let's start by importing the necessary libraries and training a simple model that we can practice saving and loading.

In [1]:
import pickle
import json
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import joblib
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

In [3]:
# Create a directory to save our model artifacts
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

In [4]:
# Load the Iris dataset
data = load_iris()
X, y = data.data, data.target
feature_names = data.feature_names
target_names = data.target_names

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size:     {X_test.shape[0]}")
print(f"Features:          {feature_names}")
print(f"Classes:           {target_names}")

Training set size: 120
Test set size:     30
Features:          ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Classes:           ['setosa' 'versicolor' 'virginica']



---

## 2. Pickle: Python's Built-in Serializer

`pickle` is Python's built-in module for serializing and deserializing any Python object into a byte stream.

#### Key characteristics:
- Part of the Python standard library (no installation needed)
- Can serialize virtually any Python object
- Uses `dump()` to write and `load()` to read
- Files are typically saved with a `.pkl` extension

> ⚠️ **Security Warning:** Pickle files can execute arbitrary code when loaded. **Never unpickle data from an untrusted source**. A malicious `.pkl` file can run harmful code the moment you call `pickle.load()`. Only load pickle files that you or your team have created.

Let's train a bare `RandomForestClassifier` and save it with pickle.

In [5]:
# Train a bare model (no pipeline, just the estimator)
# We'll need to manually scale the data first
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

# Evaluate
y_pred_original = rf_model.predict(X_test_scaled)
original_accuracy = accuracy_score(y_test, y_pred_original)
print(f"Original model accuracy: {original_accuracy:.4f}")


Original model accuracy: 1.0000


In [6]:
# Save the model with pickle
pickle_path = MODELS_DIR / "rf_model_pickle.pkl"

with open(pickle_path, "wb") as f:  # "wb" = write bytes
    pickle.dump(rf_model, f)

print(f"Model saved to: {pickle_path}")
print(f"File size:      {pickle_path.stat().st_size / 1024:.1f} KB")

Model saved to: models/rf_model_pickle.pkl
File size:      173.1 KB


In [7]:
# Load the model back from disk
with open(pickle_path, "rb") as f:  # "rb" = read bytes
    rf_model_loaded = pickle.load(f)
    
# Verify: predictions must be identical
y_pred_loaded = rf_model_loaded.predict(X_test_scaled)
print(f"Predictions match: {np.array_equal(y_pred_original, y_pred_loaded)}")

Predictions match: True



---

## 3. Joblib: The ML-Optimized Alternative

`joblib is the serialization library **recommended by scikit-learn** for persisting models. It provides two key advantages over pickle:
1. **Efficiency with NumPy arrays:** ML models often contain large NumPy arrays (e.g., the learned weights of a Random Forest's many decision trees). Joblib is specifically optimized to serialize these efficiently.
2. **Built-in compression:** Joblib natively supports compressed saving, which can significantly reduce file sizes. This is useful when deploying models to cloud services or sending them to clients.

The API is almost identical to pickle: `joblib.dump()` and `joblib.load()`

In [8]:
# Save with joblib (no compression)
joblib_path = MODELS_DIR / "rf_model_joblib.joblib"
joblib.dump(rf_model, joblib_path)

print(f"Joblib (no compression): {joblib_path.stat().st_size / 1024:.1f} KB")

Joblib (no compression): 182.5 KB


In [9]:
# Save with joblib (compressed)
# compress=3 is a good balance between speed and size (range: 0-9)
joblib_compressed_path = MODELS_DIR / "rf_model_joblib_compressed.joblib"
joblib.dump(rf_model, joblib_compressed_path, compress=3)

print(f"Joblib (compressed=3):   {joblib_compressed_path.stat().st_size / 1024:.1f} KB")

Joblib (compressed=3):   25.7 KB


In [10]:
# Load and verify
rf_model_from_joblib = joblib.load(joblib_path)
y_pred_joblib = rf_model_from_joblib.predict(X_test_scaled)

print(f"Predictions match: {np.array_equal(y_pred_original, y_pred_joblib)}")

Predictions match: True



### 3.1. File Size Comparison
Let's compare the file sized of the three approaches side by side.

In [11]:
print("File Size Comparison")
print("=" * 45)
print(f"{'Method':<30} {'Size (KB)':>10}")
print("-" * 45)
print(f"{'Pickle (.pkl)':<30} {pickle_path.stat().st_size / 1024:>10.1f}")
print(f"{'Joblib (.joblib)':<30} {joblib_path.stat().st_size / 1024:>10.1f}")
print(f"{'Joblib (compressed=3)':<30} {joblib_compressed_path.stat().st_size / 1024:>10.1f}")

File Size Comparison
Method                          Size (KB)
---------------------------------------------
Pickle (.pkl)                       173.1
Joblib (.joblib)                    182.5
Joblib (compressed=3)                25.7


### 3.2. When to Use Which?

| | Pickle | Joblib |
| :--- | :--- | :--- |
| **Best for** | General Python objects (dicts, lists, configs) | ML models with large NumPy arrays |
| **Compression** | Not built-in (requires `gzip` wrapper) | Built-in (`compress` parameter) |
| **Speed** | Slightly slower with large arrays | Optimized for large arrays |
| **Recommendation** | Use for non-ML Python objects | **Use for scikit-learn models** |

**Rule of thumb:** If it's a trained ML model, use `joblib`. If it's a plain Python dictionary or config object, `pickle` is fine.

---

## 4. The Pipeline Rule: Save the Whole Pipeline

In the previous sections, we saved a **bare model**, just the `RandomForestClassifier`. But in our training workflow, we also used a `StandardScaler` to preprocess the data. If we only save the model and forget the scaler, we have a serious problem:

The model was trained on **scaled** data, but in production, the incoming data will be **raw** (unscaled). The predictions will be garbage.

This is why you should **always persist the entire scikit-learn Pipeline**, not just the estimator. A Pipeline bundles preprocessing and the model into a single object. One file = one complete prediction unit.

> 💡 **Recall:** You already built Pipelines in the Machine Learning specialization (Pipelines and GridSearchCV lab). Now you are learning to save them, which is the bridge to production.

### 4.1. The Wrong Way (Don't Do This)

In [12]:
# ❌ THE WRONG WAY: Saving only the model
# Imagine it's "production" and we receive raw (unscaled) data
sample_raw = X_test[:5] # Raw, unscaled features

# The bare model was trained on scaled data, so feeding it raw data is wrong
y_pred_wrong = rf_model_loaded.predict(sample_raw)
y_pred_correct = rf_model_loaded.predict(scaler.transform(sample_raw))

print("Predictions with RAW data (wrong):   ", y_pred_wrong)
print("Predictions with SCALED data (right): ", y_pred_correct)
print(f"Actual labels:                        {y_test[:5]}")
print(f"\nDo they match? {np.array_equal(y_pred_wrong, y_pred_correct)}")
print("\n⚠️  The raw-data predictions may look close on this simple dataset,")
print("but on real-world data with different feature scales, this WILL break.")

Predictions with RAW data (wrong):    [2 2 2 2 2]
Predictions with SCALED data (right):  [1 0 2 1 1]
Actual labels:                        [1 0 2 1 1]

Do they match? False

⚠️  The raw-data predictions may look close on this simple dataset,
but on real-world data with different feature scales, this WILL break.


### 4.2. The Right Way: Persist the Pipeline

In [14]:
# ✅ THE RIGHT WAY: Build and save a complete pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train the pipeline on RAW data. It handles scaling internally (applies fit_transform to training data, transform to test data).
pipeline.fit(X_train, y_train)

# Evaluate the pipeline
pipeline_accuracy = accuracy_score(y_test, pipeline.predict(X_test))
print(f"Pipeline accuracy: {pipeline_accuracy:.4f}")

Pipeline accuracy: 1.0000


In [15]:
# Save the ENTIRE pipeline as one artifact
pipeline_path = MODELS_DIR / "iris_pipeline.joblib"
joblib.dump(pipeline, pipeline_path)

print(f"Pipeline saved to: {pipeline_path}")
print(f"File size:         {pipeline_path.stat().st_size / 1024:.1f} KB")

Pipeline saved to: models/iris_pipeline.joblib
File size:         183.1 KB


In [16]:
# Simulate production: Load the pipeline and predict on RAW data
loaded_pipeline = joblib.load(pipeline_path)

# Feed it raw, unscaled data. The pipeline handles everthing internally (scaling + prediction)
sample_raw = X_test[:5]
predictions = loaded_pipeline.predict(sample_raw)

print("Predictions from loaded pipeline:", predictions)
print("Actual labels:                   ", y_test[:5])
print(f"\n✅ The pipeline scales the data internally. No manual preprocessing needed.")

Predictions from loaded pipeline: [1 0 2 1 1]
Actual labels:                    [1 0 2 1 1]

✅ The pipeline scales the data internally. No manual preprocessing needed.



---

## 5. Saving Model Metadata

A model file on its own doesn't tell you much. Six months from now, you (or a client) will wonder:
- What features does this model expect?
- When was it trained?
- How well did it perform?
- What library versions were used?

**Best practice:** Save a JSON metadata file alongside every model artifact. This acts as a "birth certificate" for the model.

In [17]:
import sklearn

metadata = {
    "model_name": "iris_pipeline",
    "model_version": "1.0",
    "description": "Random Forest classifier for Iris species prediction with StandardScaler preprocessing.",
    "training_date": datetime.now(timezone.utc).isoformat(),
    "dataset": "sklearn.datasets.load_iris",
    "features": list(feature_names),
    "target_classes": list(target_names),
    "test_accuracy": round(pipeline_accuracy, 4),
    "pipeline_steps": [step[0] for step in pipeline.steps],
    "hyperparameters": {
        "n_estimators": 100,
        "random_state": 42
    },
    "library_versions": {
        "scikit-learn": sklearn.__version__,
        "joblib": joblib.__version__,
        "numpy": np.__version__
    }
}

metadata_path = MODELS_DIR / "iris_pipeline_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
    
print(f"Metadata saved to: {metadata_path}")
print()
print(json.dumps(metadata, indent=2))

Metadata saved to: models/iris_pipeline_metadata.json

{
  "model_name": "iris_pipeline",
  "model_version": "1.0",
  "description": "Random Forest classifier for Iris species prediction with StandardScaler preprocessing.",
  "training_date": "2026-02-27T14:31:45.857721+00:00",
  "dataset": "sklearn.datasets.load_iris",
  "features": [
    "sepal length (cm)",
    "sepal width (cm)",
    "petal length (cm)",
    "petal width (cm)"
  ],
  "target_classes": [
    "setosa",
    "versicolor",
    "virginica"
  ],
  "test_accuracy": 1.0,
  "pipeline_steps": [
    "scaler",
    "classifier"
  ],
  "hyperparameters": {
    "n_estimators": 100,
    "random_state": 42
  },
  "library_versions": {
    "scikit-learn": "1.7.2",
    "joblib": "1.5.2",
    "numpy": "2.3.4"
  }
}



---

## 6. Summary 

| Concept | Key Takeaway |
| :--- | :--- |
| **Pickle** | Python's built-in serializer. Works for any Python object. Use for general objects. |
| **Joblib** | Optimized for NumPy-heavy objects. **Preferred for scikit-learn models.** |
| **Compression** | `joblib.dump(model, path, compress=3)` reduces file size with minimal speed cost. |
| **Pipeline Rule** | Always save the **entire Pipeline** (preprocessing + model), not just the bare estimator. |
| **Metadata** | Save a JSON sidecar file with feature names, accuracy, versions, and training date. |
| **Security** | Never load `.pkl` / `.joblib` files from untrusted sources. |
| **Versioning** | Track `scikit-learn` and `numpy` versions. A version mismatch can break deserialization. |

---

**Next:** [ONNX Basics](./02_onnx_basics.ipynb) — Saving models in a portable, cross-platform format.